In [64]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

In [65]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

model = SentenceTransformer("sentence-transformers/all-roberta-large-v1")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [72]:
model.encode(["hej jag heter Julius", "Hej vi bygger en cool app"]).shape

(2, 1024)

In [ ]:
class Attention(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.roberta = SentenceTransformer("sentence-transformers/all-roberta-large-v1")

        self.latent = nn.Sequential(
            nn.Linear(1024, 1024),
            nn.GELU(),
            nn.Linear(1024, 1024),
        )
        
        self.similarity = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.Linear(1024, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
        
        self.priority = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.Linear(1024, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
        
    def freeze_roberta(self):
        for module in self.roberta.parameters():
            module.requires_grad = False
        
    def get_embeddings(self, sentences):
        return self.roberta.encode(sentences)

    def forward(self, x, Gx: torch.Tensor):
        x: torch.Tensor = self.get_embeddings(x)
        priority = self.priority(x)
        similarity = None

        if Gx is not None:
            K = Gx.shape[0]
            B = x.shape[0]
            x = x.unsqueeze(1).expand(-1, K, -1)
            Gx = Gx.unsqueeze(0).expand(B, -1, -1)
            xGx = torch.concat([x, Gx], dim=-1)
            similarity = self.similarity(xGx)

        return priority, similarity # (B, 1), (B, K, 1)


In [ ]:
def train():
    
    ds_priority = 
    
    model = Attention()
    model.freeze_roberta()
    
    opt = optim.AdamW(model.parameters(), lr=0.0001)
    
    